# Combinations and outcome: does a feature go with improvement differently depending on what else the patient has?

Runs on one **unit** (`UNIT`, the first code cell; see `cc_stats/units.py`): for `types` the five core groups, for `codes` the 7 codes with at least 20 patients before surgery (focal awareness codes aside). The types notebook is the source; the codes one is generated from it by `analysis/unit_notebooks.py` (only `UNIT` and the notes differ), so edit the types notebook and regenerate. Below, "type" means a group of the unit.

For each **feature** A and **context** B (the unit's groups and LGS): among patients who had B before surgery, the share who improved **with A vs without A**. For example, among LGS patients, improvement with vs without myoclonic seizures.

- This is the outcome counterpart of `conditional_disappearance.ipynb`, which asks the same question of disappearance. `1_improvement/<unit>/Firth_results.ipynb` section 2c shows the LGS column of this grid for every group.
- Cohort: patients with the outcome, LGS and every group recorded (the same patients as the Firth models). Every table and figure is shown for both outcomes, Engel then ILAE (see `cc_stats/outcomes.py`).
- Each pair: Wilson 95% CIs and Fisher's exact test, **Holm-corrected** across every pair within each outcome.

`combinations_and_outcome.py` computes the table, and this notebook runs it. The output goes to `results/<version>/4_combinations/<unit>/combinations_and_outcome.csv`.

In [ ]:
# The unit this notebook runs on: 'types' or 'codes' (cc_stats/units.py).
# Set here only; the codes notebook is generated with 'codes' (analysis/unit_notebooks.py).
UNIT = 'codes'

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from cc_stats.outcomes import OUTCOMES, heading
from cc_stats.paths import DATASET, PROJECT_ROOT, results_dir
from cc_stats.paths import ANALYSIS_CSV
from cc_stats.plots import PAIR_A, PAIR_B, pair_heatmap, paired_rate_plot, save_figure
from cc_stats.units import unit as get_unit

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/4_combinations/combinations_and_outcome.py', UNIT],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'combinations_and_outcome.py failed:\n{run.stderr}')
print('\n'.join(l for l in run.stdout.splitlines() if l.split(':')[0] in OUTCOMES))
analysis = pd.read_csv(ANALYSIS_CSV, index_col='subject_id')
unit = get_unit(UNIT, analysis)
FOLDER = f'4_combinations/{UNIT}'
all_pairs = pd.read_csv(results_dir(FOLDER) / 'combinations_and_outcome.csv')
pairs = {outcome: all_pairs.query('outcome == @outcome').reset_index(drop=True) for outcome in OUTCOMES}
FEATURES = unit.pair_groups + ['lgs']
# Heatmaps grow with the number of rows (drawn for 6: five core groups and LGS)
big = max(1, len(FEATURES) / 6)
# The myoclonic group the robustness check below looks at
MYOCLONIC = 'myoclonic' if UNIT == 'types' else 'code_05'


def fmt_p(v):
    return '<0.001' if v < 0.001 else f'{v:.3f}'


def pct(k, n):
    return f'{k}/{n} ({k / n:.0%})' if n else '-'

## Every pair

Each cell: improvement **with** the row's feature minus **without** it, among patients with the column's context, in percentage points. Red means patients with the feature improved more often. Cells with fewer than 5 patients on either side are greyed out; the diagonal is empty.

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    p = pairs[outcome]
    fig, ax = plt.subplots(figsize=(10 * big, 6.5 * big))
    pair_heatmap(p, FEATURES, FEATURES, ax=ax, row='feature', col='context',
                 row_label='with {}', col_label='among {}', counts=('improved_with', 'improved_without'),
                 cbar_label=f"Difference in improvement, {O['improved_label']} (percentage points)")
    survives = (p['p_holm'] < 0.05).any()
    ax.set_title(f"{O['name']}: improvement with vs without each feature, within each context\n"
                 f"(* uncorrected p < 0.05; {'some survive' if survives else 'none survive'} Holm)", loc='left')
    plt.tight_layout()
    save_figure(FOLDER, f'combinations_and_outcome_heatmap_{outcome}')
    plt.show()

## Strongest pairs

Sorted by uncorrected p. The Holm column is the one to trust.

In [ ]:
for outcome in OUTCOMES:
    heading(outcome)
    top = pairs[outcome].sort_values('p').head(8)
    display(pd.DataFrame({
        'Feature': top['feature'],
        'Among': top['context'],
        'With feature': [pct(k, n) for k, n in top[['improved_with', 'n_with']].to_numpy()],
        'Without': [pct(k, n) for k, n in top[['improved_without', 'n_without']].to_numpy()],
        'Difference': (top['difference'] * 100).round().astype(int).astype(str) + ' pts',
        'p': top['p'].map(fmt_p),
        'Holm p': top['p_holm'].map(fmt_p),
    }).style.hide(axis='index'))

## The six strongest pairs

Observed improvement with vs without the feature, within the context, with Wilson 95% CIs.

In [ ]:
for outcome, O in OUTCOMES.items():
    heading(outcome)
    # paired_rate_plot draws the first row at the bottom, so the strongest goes last
    top = pairs[outcome].sort_values('p').head(6).iloc[::-1]
    labels = top['feature'] + ', among ' + top['context']
    fig, ax = plt.subplots(figsize=(11, 5))
    paired_rate_plot(ax, labels, top=('with feature', top['improved_with'], top['n_with'], PAIR_A),
                     bottom=('without feature', top['improved_without'], top['n_without'], PAIR_B),
                     p=top['p'], p_holm=top['p_holm'])
    ax.set_xlabel(f"Observed improvement, {O['improved_label']} (Wilson 95% CI)")
    ax.set_title(f"{O['name']}: the six strongest pairs", loc='left', pad=28)
    ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, handletextpad=0.3)
    plt.tight_layout()
    save_figure(FOLDER, f'combinations_and_outcome_strongest_{outcome}')
    plt.show()

## Robustness: myoclonic within a stricter LGS group

12 of the 47 LGS patients have no tonic code before surgery, although tonic seizures are required for LGS (see `0_cohort/codes/code_counts.ipynb`). The stricter group is the same patients for both units: LGS with generalized or focal tonic (codes 4 or 23, the tonic type). The myoclonic group is the myoclonic type for `types` and generalized myoclonic (code 5) for `codes`. Restricting LGS to patients who do have a tonic code checks that the myoclonic-within-LGS result is not driven by the patients whose LGS label fits least well.

In [ ]:
import scipy.stats as st

core = [unit.pre(t) for t in unit.pair_groups]
rows = []
for outcome, O in OUTCOMES.items():
    cohort = analysis.dropna(subset=[O['improved'], 'lgs'] + core)
    for name, lgs in (('all LGS', cohort[cohort['lgs'] == 1]),
                      ('LGS with a tonic code', cohort[(cohort['lgs'] == 1) & (cohort['has_tonic'] == 1)])):
        improved = lgs[O['improved']]
        with_m, without_m = improved[lgs[unit.pre(MYOCLONIC)] == 1], improved[lgs[unit.pre(MYOCLONIC)] == 0]
        table = [[int(with_m.sum()), int((with_m == 0).sum())], [int(without_m.sum()), int((without_m == 0).sum())]]
        rows.append({'Outcome': O['name'], 'Group': name, 'With myoclonic': pct(int(with_m.sum()), len(with_m)),
                     'Without': pct(int(without_m.sum()), len(without_m)), 'Fisher p': fmt_p(st.fisher_exact(table)[1])})
display(pd.DataFrame(rows).style.hide(axis='index'))

## Reading the results

Checked 2026-10-21 on v2 (Engel n = 95, 16 not improved; ILAE n = 88, 23 not improved), over the 7 codes with at least 20 patients and LGS: 56 pairs per outcome.

- **Nothing survives Holm** on either outcome. With 56 comparisons and so few non-improvers, this is exploratory.
- **Generalized myoclonic (5) tops both lists**, as the myoclonic type does. It gives the largest difference among generalized tonic-clonic, tonic and atonic and among LGS on both outcomes (and among spasms on ILAE), though not among focal tonic or focal to BTC: among LGS (ILAE 14/14 vs 15/29, p = 0.001, Holm 0.073, the closest to surviving; Engel 15/15 vs 22/30, p = 0.038), among generalized tonic-clonic (Engel 14/14 vs 17/25; ILAE 12/12 vs 13/23), among generalized tonic (ILAE 14/15 vs 13/22) and among generalized atonic (Engel 9/9 vs 14/22).
- **Within the stricter LGS group** (LGS with a tonic code), it holds and is a little clearer than for the type: Engel 11/11 vs 17/22 (p = 0.14), ILAE 11/11 vs 11/21 (p = 0.006); the type gives 11/12 vs 11/20 (p = 0.050) on ILAE.
- **Generalized atonic within LGS goes the other way**, as the atonic type does: Engel 12/18 vs 25/27 (p = 0.045).
- **The pairs are not independent**: the contexts overlap and the same few non-improvers appear in many cells.
- **The clinical question for the team stands:** LGS with generalized myoclonic seizures may partly be a distinct syndrome (e.g. myoclonic-atonic epilepsy), which the dataset doesn't record beyond the LGS flag.